# CNNs avec Keras sur des données de paysage

Dans ces travaux pratiques, nous classons les photos du jeu des paysages (Intel Image Classification, 6 classes) :

1. avec le CNN des slides, puis un CNN plus profond, que vous écrivez ;
2. en augmentant les données par des couches ;
3. avec trois architectures historiques : LeNet, AlexNet et un bloc Inception ;
4. en analysant les erreurs sur l'ensemble de test ;
5. par transfert d'apprentissage : EfficientNetV2B0 pré-entraîné sur ImageNet, gelé puis fine-tuné (partie « Transfert d'apprentissage ») ;
6. avec un *Vision Transformer* pré-entraîné (partie « Transfert d'apprentissage avec des transformeurs »).

Les parties 5 et 6 vont avec le chapitre « Transfer learning » des slides : arrêtez-vous avant elles si vous n'en êtes qu'au chapitre sur les CNN.

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'exécution` et vérifiez que l'accélérateur matériel est un GPU : sans lui, chaque entraînement de ce TP prendrait des heures.

In [ ]:
!nvidia-smi

## Téléchargement du jeu des paysages

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Imports

In [ ]:
!pip install -q keras-hub
import itertools
import pathlib

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.metrics
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Préparation des données

Comme dans les slides, `keras.utils.image_dataset_from_directory` lit les images de `seg_train` par batchs de 32, dossier par dossier : l'étiquette d'une image est l'indice de son dossier, dans l'ordre alphabétique (0 = `buildings`, …, 5 = `street`). 20 % des images servent à la validation, avec un découpage fixé par `seed=0`.

Les images restent sur le disque et sont lues à chaque époque. En mémoire, en flottants de 4 octets, les 14 034 images d'entraînement prendraient 3,8 Go en 150 × 150 et 8,5 Go en 224 × 224 (la taille du transfert d'apprentissage) : plus que ce que Colab laisse libre à côté du modèle, sur ses 12,7 Go de RAM.

L'ensemble de test (`seg_test`) est lu sans mélange (`shuffle=False`) : les prédictions du modèle sortent dans l'ordre de `test_ds.file_paths`, celui de `test_labels`.

In [ ]:
def load_landscapes(image_size: tuple[int, int]) -> tuple:
  """Charge les images d'entraînement, de validation et de test à cette taille."""
  train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "landscape/seg_train",
    image_size=image_size,
    validation_split=0.2,  # 20 % des images pour la validation
    subset="both",
    seed=0,  # le même découpage à chaque exécution
  )
  test_ds = keras.utils.image_dataset_from_directory(
    "landscape/seg_test", image_size=image_size, shuffle=False
  )
  return train_ds, val_ds, test_ds


train_ds, val_ds, test_ds = load_landscapes((150, 150))
label_names = train_ds.class_names
# Chemins et étiquettes des images de test, dans l'ordre des prédictions
test_paths = numpy.array(test_ds.file_paths)
test_labels = numpy.concatenate([labels for _, labels in test_ds])
print(label_names)

## Regardons les données

In [ ]:
# Un batch : 32 images 150 × 150 à 3 canaux, et leurs 32 étiquettes
images, labels = next(iter(train_ds))
print(f"Forme d'un batch d'images : {images.shape}")
print(f"Forme d'un batch d'étiquettes : {labels.shape}")

counts = [
  len(list((pathlib.Path("landscape/seg_train") / name).iterdir()))
  for name in label_names
]
plt.bar(label_names, counts)
plt.title("Nombre d'images par classe (entraînement et validation)")
plt.ylabel("Nombre d'images")
plt.xlabel("Classe")
plt.show()

In [ ]:
# Les pixels sont des flottants de 0 à 255 : imshow les attend en entiers
images = numpy.asarray(images).astype("uint8")
_, ax = plt.subplots(5, 5, figsize=(15, 15))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  ax[i, j].imshow(images[k])
  ax[i, j].set_title(label_names[int(labels[k])])
  ax[i, j].axis("off")
plt.show()

## Création du modèle

Le CNN des slides (« Définir un CNN ») : deux blocs convolution + pooling, puis un pooling global qui garde une valeur par canal, et une couche dense de sortie. `Rescaling(1 / 255)` ramène les pixels de 0–255 à 0–1, l'échelle où un réseau entraîné depuis zéro apprend bien.

In [ ]:
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),  # pixels 0–255 → 0–1
    layers.Conv2D(32, 3, activation="relu"),  # 32 filtres 3 × 3
    layers.MaxPooling2D(2),  # fenêtres 2 × 2, pas de 2
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.GlobalAveragePooling2D(),  # une valeur par canal
    layers.Dense(6, activation="softmax"),  # une probabilité par classe
  ]
)
model.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-3),
  loss="sparse_categorical_crossentropy",  # étiquettes entières
  metrics=["accuracy"],
)
model.summary()

## Pouvez-vous expliquer les différents nombres de paramètres ?

Retrouvez par le calcul le nombre de paramètres de chaque couche du résumé. Pourquoi certaines couches n'en ont-elles aucun ?

### Solution

- `conv2d` : chacun des 32 filtres a 3 × 3 poids par canal d'entrée, sur 3 canaux (RVB), plus un biais : (3 × 3 × 3 + 1) × 32 = 896.
- `conv2d_1` : 64 filtres, qui couvrent chacun les 32 canaux de la couche précédente : (3 × 3 × 32 + 1) × 64 = 18 496.
- `dense` : 64 entrées (une par canal, après le pooling global) et 6 sorties, plus 6 biais : 64 × 6 + 6 = 390.
- `rescaling`, les poolings et le pooling global n'apprennent rien : ils appliquent une opération fixe (multiplier, prendre un maximum ou une moyenne).

Total : 19 782 paramètres. Le pooling global garde la tête petite : avec `Flatten`, la couche dense recevrait 36 × 36 × 64 = 82 944 valeurs, soit 497 670 paramètres.

## Apprentissage

Entraînons ce modèle comme dans les slides (« Entraîner avec arrêt précoce ») : au plus 10 époques, arrêtées après 3 époques sans baisse de la perte de validation, avec retour aux meilleurs poids.

Un callback garde son état d'un `fit` à l'autre. Avec Keras 3.13, la version de Colab, un même `EarlyStopping` repart de la meilleure perte de validation de l'entraînement précédent : il arrêterait le modèle suivant au bout de 3 époques et lui rendrait les poids de sa première époque. `early_stopping()` en crée donc un neuf pour chaque entraînement.

`train(model, epochs, learning_rate)` regroupe ensuite ces étapes (compilation, entraînement avec arrêt précoce, courbes) pour les modèles suivants.

In [ ]:
def early_stopping() -> keras.callbacks.EarlyStopping:
  """Un arrêt précoce neuf : 3 époques de patience, retour aux meilleurs poids."""
  return keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
  )


history = model.fit(
  train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping()]
)


def plot_metrics(history: keras.callbacks.History) -> None:
  """Trace la justesse et la perte, en entraînement et en validation."""
  for metric, name in (("accuracy", "Justesse"), ("loss", "Perte")):
    plt.plot(history.history[metric], label="Entraînement")
    plt.plot(history.history[f"val_{metric}"], label="Validation")
    plt.title(f"{name} du modèle")
    plt.ylabel(name)
    plt.xlabel("Époque")
    plt.legend()
    plt.show()


plot_metrics(history)


def train(
  model: keras.Model, epochs: int, learning_rate: float = 1e-4
) -> keras.callbacks.History:
  """Compile et entraîne un modèle sur les images 150 × 150, puis trace ses courbes."""
  model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  history = model.fit(
    train_ds, validation_data=val_ds, epochs=epochs, callbacks=[early_stopping()]
  )
  plot_metrics(history)
  return history

## Améliorez cette performance

Le CNN des slides plafonne vers 66 % de justesse : deux convolutions ne voient que de petits motifs, trop peu pour reconnaître une scène. Écrivez une fonction `deep_cnn()` qui renvoie un CNN plus profond, avec des poids neufs à chaque appel, puis entraînez-le avec `train` :

- des blocs convolution + pooling répétés, jusqu'à des cartes de quelques pixels de côté ;
- du *dropout* avant les couches denses de la tête, contre le surapprentissage ;
- entre 10 et 20 époques, chacune en moins d'une minute sur le GPU de Colab.

Objectif : plus de 70 % de justesse en validation. La solution atteint environ 86 % en 15 époques d'environ 45 secondes. `deep_cnn()` resservira pour l'augmentation de données.

**Documentation utile** :

- https://keras.io/api/layers/convolution_layers/convolution2d/
- https://keras.io/api/layers/pooling_layers/max_pooling2d/
- https://keras.io/api/layers/reshaping_layers/flatten/
- https://keras.io/api/layers/regularization_layers/dropout/

In [ ]:
def deep_cnn() -> keras.Sequential: ...


# Votre code ici

### Conseils supplémentaires

- Entrée : `keras.Input(shape=(150, 150, 3))` puis `layers.Rescaling(1 / 255)`, comme le CNN des slides.
- Un bloc : `layers.Conv2D(200, 3, activation="relu", padding="same", kernel_initializer="orthogonal")` puis `layers.MaxPooling2D(2, padding="same")`. Avec `padding="same"`, la convolution garde la taille des cartes et seul le pooling la divise par 2, arrondie au-dessus : 150 → 75 → 38 → 19 → 10 → 5 → 3 en 6 blocs. Une 7ᵉ convolution, sans pooling, termine sur des cartes 3 × 3.
- Tête : `layers.Flatten()`, puis trois couches `Dense` ReLU de 200, 100 et 50 neurones, et la sortie `layers.Dense(6, activation="softmax")`, chacune précédée de `layers.Dropout(0.2)`.
- Entraînement : `train(model, epochs=15)`, avec le taux d'apprentissage par défaut de `train` (1e-4).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def deep_cnn() -> keras.Sequential:
  blocks = []
  for _ in range(6):
    blocks += [layers.Conv2D(...), layers.MaxPooling2D(...)]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      layers.Conv2D(...),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      ...,
      layers.Dense(6, activation="softmax"),
    ]
  )


model = deep_cnn()
model.summary()
history = train(model, epochs=...)
```

### Solution

In [ ]:
def deep_cnn() -> keras.Sequential:
  """Le CNN profond de la solution, avec des poids neufs à chaque appel."""
  blocks = []
  for _ in range(6):  # cartes de 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels de côté
    blocks += [
      layers.Conv2D(
        200,
        3,
        activation="relu",
        padding="same",
        kernel_initializer="orthogonal",
      ),
      layers.MaxPooling2D(2, padding="same"),
    ]
  return keras.Sequential(
    [
      keras.Input(shape=(150, 150, 3)),
      layers.Rescaling(1 / 255),
      *blocks,
      layers.Conv2D(
        200, 3, activation="relu", padding="same", kernel_initializer="orthogonal"
      ),
      layers.Flatten(),
      layers.Dropout(0.2),
      layers.Dense(200, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(100, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(50, activation="relu"),
      layers.Dropout(0.2),
      layers.Dense(6, activation="softmax"),
    ]
  )


model = deep_cnn()
model.summary()

In [ ]:
history = train(model, epochs=15)

## Augmentation de données

Quand la justesse d'entraînement continue de monter alors que celle de validation stagne, le modèle surapprend. L'augmentation de données (slide « Augmenter les données par des couches ») lui présente à chaque époque des variantes des images d'entraînement : `layers.RandomFlip("horizontal")` retourne une image une fois sur deux, et `layers.RandomZoom((-0.3, 0))` zoome dessus jusqu'à 30 %. Ces couches n'agissent que pendant `fit` : `evaluate` et `predict` voient les images intactes.

Placez ces deux couches devant la même architecture, avec des poids neufs (`deep_cnn()`), puis entraînez 15 époques. L'écart entre les courbes d'entraînement et de validation se réduit-il ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un modèle Keras s'emploie comme une couche : `keras.Sequential` accepte `deep_cnn()` dans sa liste, après `keras.Input(shape=(150, 150, 3))` et les deux couches d'augmentation. Les valeurs négatives de `RandomZoom` zooment en avant. Entraînement : `train(model, epochs=15)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.RandomFlip(...),
    layers.RandomZoom(...),
    ...,
  ]
)
model.summary()
history = train(model, epochs=15)
```

### Solution

In [ ]:
model = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.RandomFlip("horizontal"),  # miroir, une image sur deux
    layers.RandomZoom((-0.3, 0)),  # zoom avant, jusqu'à 30 %
    deep_cnn(),
  ]
)
model.summary()
history = train(model, epochs=15)

Chaque époque voit des variantes différentes des images : la justesse d'entraînement monte moins vite et l'écart avec la validation se réduit. Le modèle doit apprendre des caractéristiques qui résistent au cadrage et à l'orientation, au lieu de mémoriser les images d'entraînement. L'évaluation sur le test et l'analyse d'erreur portent sur ce modèle, `model`.

## Architectures historiques

Trois architectures du chapitre sur les CNN, à entraîner puis comparer au CNN profond : leur code est donné. Chacune a son propre nom de variable, et `model` reste le CNN profond augmenté.

### LeNet

[LeNet-5](https://en.wikipedia.org/wiki/LeNet) (1998), conçu pour des chiffres manuscrits de 32 × 32 pixels : des convolutions 5 × 5, des activations sigmoïdes et des poolings par moyenne.

In [ ]:
le_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(6, 5, padding="same", activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Conv2D(16, 5, activation="sigmoid"),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(120, activation="sigmoid"),
    layers.Dense(84, activation="sigmoid"),
    layers.Dense(6, activation="softmax"),
  ],
  name="le_net",
)
le_net.summary()
le_net_history = train(le_net, epochs=15)

### AlexNet

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) (2012), vainqueur d'ImageNet : une première convolution 11 × 11 de pas 4, des ReLU, du *dropout* dans une tête de deux couches de 4 096 neurones.

In [ ]:
alex_net = keras.Sequential(
  [
    keras.Input(shape=(150, 150, 3)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(96, 11, strides=4, activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(256, 5, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(384, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(3, strides=2),
    layers.Flatten(),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(4096, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(6, activation="softmax"),
  ],
  name="alex_net",
)
alex_net.summary()
alex_net_history = train(alex_net, epochs=15)

### Bloc Inception

Un bloc [Inception](https://en.wikipedia.org/wiki/Inception_(deep_learning_architecture)) (GoogLeNet, 2014) : quatre branches en parallèle (convolutions 1 × 1, 3 × 3, 5 × 5 et pooling), concaténées sur l'axe des canaux. Un réseau complet en empile plusieurs. Les branches parallèles demandent l'[API fonctionnelle de Keras](https://keras.io/guides/functional_api/), où chaque couche s'applique à un tenseur.

In [ ]:
def inception_block(inputs: keras.KerasTensor, filters: int = 64) -> keras.KerasTensor:
  # conv 1 × 1
  branch_1 = layers.Conv2D(filters, 1, activation="relu")(inputs)

  # conv 1 × 1 puis conv 3 × 3
  branch_3 = layers.Conv2D(filters, 1, activation="relu")(inputs)
  branch_3 = layers.Conv2D(filters, 3, padding="same", activation="relu")(branch_3)

  # conv 1 × 1 puis conv 5 × 5
  branch_5 = layers.Conv2D(filters, 1, activation="relu")(inputs)
  branch_5 = layers.Conv2D(filters, 5, padding="same", activation="relu")(branch_5)

  # max pooling 3 × 3 de pas 1 puis conv 1 × 1
  branch_pool = layers.MaxPooling2D(3, strides=1, padding="same")(inputs)
  branch_pool = layers.Conv2D(filters, 1, activation="relu")(branch_pool)

  # concaténation sur l'axe des canaux : 4 × 64 = 256 canaux
  return layers.Concatenate()([branch_1, branch_3, branch_5, branch_pool])


inputs = keras.Input(shape=(150, 150, 3))
x = layers.Rescaling(1 / 255)(inputs)
x = inception_block(x)
x = layers.Flatten()(x)
outputs = layers.Dense(6, activation="softmax")(x)
inception = keras.Model(inputs, outputs, name="inception")
inception.summary()
inception_history = train(inception, epochs=10)

## Évaluation des performances sur l'ensemble de test

Le dossier `seg_test` contient 3 000 images que le modèle n'a vues ni à l'entraînement, ni pour l'arrêt précoce : sa justesse sur elles estime honnêtement celle qu'il aura sur de nouvelles photos. `test_accuracies` garde la justesse en test de chaque approche, pour les comparer à la fin du TP.

In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Justesse en test : {accuracy:.3f}")
test_accuracies = {"CNN profond, données augmentées": accuracy}

## Analyse d'erreur

On affiche la matrice de confusion, puis des images mal classées.

In [ ]:
def analyze_preds(preds: numpy.ndarray, labels: numpy.ndarray) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(
    confusion_matrix,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=label_names,
    yticklabels=label_names,
  )
  plt.title("Matrice de confusion")
  plt.xlabel("Classe prédite")
  plt.ylabel("Classe réelle")
  plt.show()

  seaborn.countplot(x=[label_names[pred] for pred in preds])
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Classe")
  plt.show()


test_pred = model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  """Affiche jusqu'à 25 images de test de true_class prédites predicted_class."""
  print(f"Prédiction : {predicted_class}, classe réelle : {true_class}")
  mistakes = test_paths[
    (test_pred == label_names.index(predicted_class))
    & (test_labels == label_names.index(true_class))
  ]
  mistakes = numpy.random.choice(mistakes, size=min(len(mistakes), 25), replace=False)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  grid_indexes = list(itertools.product(range(5), repeat=2))
  for i, j in grid_indexes:
    ax[i, j].axis("off")
  for path, (i, j) in zip(mistakes, grid_indexes):
    ax[i, j].imshow(keras.utils.load_img(path))
  plt.show()

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées montagne
plot_mistakes("glacier", "mountain")

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées mer
plot_mistakes("glacier", "sea")

In [ ]:
# Images prédites bâtiments alors qu'elles sont étiquetées rue
plot_mistakes("buildings", "street")

## Transfert d'apprentissage

On réutilise un réseau pré-entraîné sur ImageNet, EfficientNetV2B0 (`keras.applications`), comme extracteur de caractéristiques : ses poids sont gelés, et on n'entraîne qu'une nouvelle tête de classification pour nos 6 classes (slides du chapitre « Transfer learning »).

Les poids pré-entraînés attendent les images telles qu'à leur entraînement :

- **Taille** : 224 × 224, la taille des images d'ImageNet sur lesquelles les poids ont appris. On recharge les paysages à cette taille.
- **Pixels de 0 à 255** : EfficientNetV2 les normalise lui-même, dans sa première couche. Le modèle ne commence donc pas par `Rescaling(1 / 255)`, contrairement aux CNN entraînés depuis zéro : il ne verrait que des images presque noires.

`base.trainable = False` exclut les poids d'EfficientNetV2 de la descente de gradient. Ses couches `BatchNormalization` deviennent alors non entraînables : elles normalisent avec les statistiques d'ImageNet, sans les mettre à jour sur nos batchs de paysages.

In [ ]:
TRANSFER_SIZE = (224, 224)  # taille d'entraînement des poids ImageNet
# Mêmes images, dans le même ordre : test_paths et test_labels restent valables
train_ds, val_ds, test_ds = load_landscapes(TRANSFER_SIZE)

In [ ]:
base = keras.applications.EfficientNetV2B0(
  include_top=False,  # sans la tête ImageNet (1000 classes)
  weights="imagenet",
  pooling="avg",  # GlobalAveragePooling2D : un vecteur de 1280 valeurs par image
)
base.trainable = False  # poids figés, BatchNorm en inférence

inputs = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = base(inputs)
outputs = layers.Dense(6, activation="softmax")(x)  # 6 paysages
model = keras.Model(inputs, outputs)
model.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(
  train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping()]
)
plot_metrics(history)

In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Justesse en test : {accuracy:.3f}")
test_accuracies["EfficientNetV2B0 gelé"] = accuracy

test_pred = model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "street")

### Questions

1. Combien de paramètres ce modèle entraîne-t-il, et combien en gèle-t-il ? (voir son résumé)
2. Comparez sa justesse de test à celle du CNN profond entraîné depuis zéro, et le temps de leurs entraînements.
3. Les erreurs restantes (matrice de confusion, images mal classées) ressemblent-elles à celles du CNN ?

#### Solution

1. Seule la tête est entraînée : 1 280 × 6 + 6 = 7 686 paramètres, contre 5 919 312 gelés dans EfficientNetV2B0.
2. Le modèle pré-entraîné dépasse le CNN profond, 93 % contre 86 % environ en test, dès les premières époques et en quelques secondes par époque : seule la tête apprend. Les caractéristiques qu'EfficientNetV2 a apprises sur plus d'un million d'images d'ImageNet se transfèrent bien à nos 14 000 paysages.
3. Les confusions qui restent opposent surtout des classes réellement proches (glacier et montagne, bâtiments et rue) ; beaucoup de ces images sont ambiguës, voire mal annotées.

### Fine-tuning : dégeler le backbone

Une fois la tête entraînée, on peut dégeler EfficientNetV2 et entraîner tout le réseau quelques époques de plus, avec un taux d'apprentissage 100 fois plus faible (slide « Fine-tuning : dégeler le backbone ») : ses poids sont déjà bons, on veut seulement les ajuster à nos paysages.

Dégelez `base`, mais gardez ses couches `BatchNormalization` gelées, recompilez `model` avec Adam à 1e-5, puis entraînez 3 époques. Ajoutez sa justesse en test à `test_accuracies`. La justesse de validation progresse-t-elle ?

Keras applique un seul taux d'apprentissage à tout le modèle : la tête, déjà entraînée, avance elle aussi à 1e-5.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `base.trainable = True` rend toutes ses couches entraînables, BatchNorm comprises. Parcourez ensuite `base.layers` et remettez `trainable = False` sur celles pour qui `isinstance(layer, layers.BatchNormalization)` est vrai. Sinon, `fit`, qui exécute tout le graphe en mode entraînement, remplacerait les statistiques d'ImageNet par celles de nos batchs et dégraderait les caractéristiques apprises.
- `model.compile(...)` est obligatoire : sans lui, le changement de `trainable` est ignoré et `fit` n'entraîne toujours que la tête.
- `model.fit(train_ds, validation_data=val_ds, epochs=3)`, puis `plot_metrics` et `model.evaluate(test_ds, verbose=0)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
base.trainable = ...
for layer in base.layers:
  if isinstance(layer, ...):
    layer.trainable = ...

model.compile(
  optimizer=keras.optimizers.Adam(...),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=...)
plot_metrics(history)
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Justesse en test : {accuracy:.3f}")
test_accuracies["EfficientNetV2B0 fine-tuné"] = accuracy
```

#### Solution

In [ ]:
base.trainable = True
# BatchNorm gelées : statistiques d'ImageNet conservées
for layer in base.layers:
  if isinstance(layer, layers.BatchNormalization):
    layer.trainable = False

# À refaire après tout changement de trainable
model.compile(
  optimizer=keras.optimizers.Adam(1e-5),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=3)
plot_metrics(history)
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Justesse en test : {accuracy:.3f}")
test_accuracies["EfficientNetV2B0 fine-tuné"] = accuracy

La justesse gagne encore un point environ, de 93 à 94 % en test : le backbone s'adapte à nos paysages. Un taux d'apprentissage trop élevé détruirait au contraire ce qu'il a appris sur ImageNet. La suite du TP utilise ce modèle.

## Prédire dans des conditions « réelles »

Le dossier `seg_pred` contient des images sans étiquette : on ne peut pas y mesurer la justesse. On peut en revanche afficher des photos et les probabilités que le modèle attribue à chaque classe.

In [ ]:
pred_paths = sorted(pathlib.Path("landscape/seg_pred").rglob("*.jpg"))
print(f"{len(pred_paths)} images sans étiquette")

In [ ]:
# 25 images au hasard, sans remise, chargées à la taille du modèle
indexes = numpy.random.choice(len(pred_paths), size=25, replace=False)
pred_images = numpy.stack(
  [
    keras.utils.img_to_array(
      keras.utils.load_img(pred_paths[index], target_size=TRANSFER_SIZE)
    )
    for index in indexes
  ]
)
probabilities = model.predict(pred_images, verbose=0)

# Chaque image, puis la distribution de ses prédictions sur la ligne d'en dessous
_, ax = plt.subplots(10, 5, figsize=(30, 45))
for k, (i, j) in enumerate(itertools.product(range(5), repeat=2)):
  predicted_class = label_names[probabilities[k].argmax()]
  ax[i * 2, j].imshow(pred_images[k].astype("uint8"))
  ax[i * 2, j].set_title(f"{pred_paths[indexes[k]].name} ({predicted_class})")
  ax[i * 2, j].axis("off")
  ax[i * 2 + 1, j].bar(label_names, probabilities[k])
plt.show()

## Justesse en fonction de la probabilité maximale

La justesse varie-t-elle selon la probabilité maximale que rend le modèle ? Si oui, on peut ne garder que les prédictions sûres, et confier les autres à un humain.

In [ ]:
test_probabilities = model.predict(test_ds, verbose=0)

In [ ]:
def threshold_accuracy(
  probabilities: numpy.ndarray, labels: numpy.ndarray, threshold: float
) -> tuple[float, float, float, float]:
  predictions = probabilities.argmax(axis=-1)
  mask_above = probabilities.max(axis=-1) > threshold
  mask_below = ~mask_above

  n_above = mask_above.sum()
  n_below = mask_below.sum()

  if n_above:
    above = (predictions[mask_above] == labels[mask_above]).sum() / n_above
  else:
    above = 1.0
  if n_below:
    below = (predictions[mask_below] == labels[mask_below]).sum() / n_below
  else:
    below = 1.0
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
  test_probabilities, test_labels, 0.99
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité au "
  f"dessus du seuil ({ratio_above * 100:.2f}% des données) : "
  f"{accuracy_above:.2f}"
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité en "
  f"dessous du seuil ({ratio_below * 100:.2f}% des données) : "
  f"{accuracy_below:.2f}"
)

xs = numpy.arange(101) / 100
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities, test_labels, x)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Justesse")
plt.plot(xs, ratios_above, label="Proportion de prédictions retenues")
plt.xlabel("Seuil de prédiction")
plt.title("Justesse et proportion de prédictions retenues selon le seuil")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

Même démarche avec un *Vision Transformer* (ViT) pré-entraîné sur ImageNet-21k (14 millions d'images), chargé avec KerasHub. Le ViT découpe une image 224 × 224 en 196 carrés de 16 × 16 pixels, et ajoute devant eux un token de classification, `[CLS]`. Le dernier état de ce token résume l'image : la tête de classification se branche sur lui.

Construisez `vit_model` :

- le préréglage `"vit_base_patch16_224_imagenet21k"`, chargé deux fois : `keras_hub.layers.ViTImageConverter`, ses prétraitements (redimensionner en 224 × 224, ramener les pixels de 0–255 à [-1, 1]), et `keras_hub.models.ViTBackbone`, le réseau, à geler ;
- en API fonctionnelle : entrée 224 × 224 × 3, prétraitements, ViT, token `[CLS]`, puis une couche dense de sortie à 6 classes ;
- compilé comme le modèle EfficientNetV2.

**Documentation utile** :

- https://keras.io/keras_hub/api/models/vit/vit_backbone/
- https://keras.io/keras_hub/api/models/vit/vit_image_converter/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Chargement : `keras_hub.layers.ViTImageConverter.from_preset(PRESET)` et `keras_hub.models.ViTBackbone.from_preset(PRESET)` (330 Mo à télécharger).
- Le convertisseur s'emploie comme une couche, juste après `keras.Input(shape=(224, 224, 3))`.
- Le ViT renvoie un vecteur de 768 valeurs par token, de forme `(None, 197, 768)` : le token `[CLS]` est le premier, `x[:, 0]`.
- `vit.trainable = False` suffit : le ViT normalise avec des `LayerNormalization`, qui n'ont pas de statistiques à figer, contrairement aux BatchNorm.
- Compilation : `keras.optimizers.Adam(1e-3)`, `loss="sparse_categorical_crossentropy"`, `metrics=["accuracy"]`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
PRESET = "vit_base_patch16_224_imagenet21k"
converter = keras_hub.layers.ViTImageConverter.from_preset(PRESET)
vit = ...
vit.trainable = ...

inputs = keras.Input(shape=(224, 224, 3))
x = converter(inputs)
x = ...
x = x[...]
outputs = ...
vit_model = keras.Model(inputs, outputs, name="vit_model")
vit_model.compile(...)
vit_model.summary()
```

### Solution

In [ ]:
PRESET = "vit_base_patch16_224_imagenet21k"
converter = keras_hub.layers.ViTImageConverter.from_preset(PRESET)
vit = keras_hub.models.ViTBackbone.from_preset(PRESET)
vit.trainable = False

inputs = keras.Input(shape=(224, 224, 3))  # pixels 0–255
x = converter(inputs)  # 224 × 224, pixels dans [-1, 1]
x = vit(x)  # (None, 197, 768) : un vecteur par token
x = x[:, 0]  # le token [CLS], qui résume l'image
outputs = layers.Dense(6, activation="softmax")(x)
vit_model = keras.Model(inputs, outputs, name="vit_model")
vit_model.compile(
  optimizer=keras.optimizers.Adam(1e-3),
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"],
)
vit_model.summary()

### Comparaison des trois approches

Entraînez `vit_model` 5 époques (environ une minute chacune sur le GPU de Colab), évaluez-le sur l'ensemble de test, puis comparez les trois approches : CNN entraîné depuis zéro, EfficientNetV2B0 pré-entraîné, ViT pré-entraîné. Qu'apporte le transfert d'apprentissage, et à quel prix ?

In [ ]:
history = vit_model.fit(
  train_ds, validation_data=val_ds, epochs=5, callbacks=[early_stopping()]
)
plot_metrics(history)

In [ ]:
loss, accuracy = vit_model.evaluate(test_ds, verbose=0)
test_accuracies["ViT gelé"] = accuracy

test_pred = vit_model.predict(test_ds, verbose=0).argmax(axis=-1)
analyze_preds(test_pred, test_labels)

for name, accuracy in test_accuracies.items():
  print(f"{name} : {accuracy:.3f}")

#### Solution

- Les deux modèles pré-entraînés dépassent nettement le CNN entraîné depuis zéro (93 à 94 % contre 86 % environ), pour une fraction de son temps d'entraînement : c'est tout l'intérêt du transfert d'apprentissage quand on a peu de données (14 000 images ici, contre plus d'un million pour ImageNet et 14 millions pour ImageNet-21k).
- Entre EfficientNetV2B0 et le ViT, l'écart est faible, mais le ViT est bien plus gros : 86 millions de paramètres contre 5,9 millions, et chaque prédiction coûte en proportion. Le fine-tuning du backbone apporte encore un point.
- Les confusions restantes sont les mêmes pour les trois modèles : glacier et montagne, bâtiments et rue.